In [3]:
from IPython.display import Image
import pandas as pd
import numpy as np
from pandas.conftest import axis_1
from scipy.signal import square

In [4]:
data = pd.read_csv('ks.csv')

In [5]:
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.00,600.00
1,Arcade County (Canceled),Games,Games,USD,2012-04-29,2012-03-30 23:40:45,canceled,5,US,71.00,9000.00
2,Hayashi Skate Co. Solar Skateboard backpack,Accessories,Fashion,CAD,2017-07-22,2017-05-23 23:00:13,canceled,8,CA,360.36,2391.77
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.00,10000.00
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.00,1400.00


In [6]:
data.shape

(378661, 11)

In [7]:
data['Состояние'].value_counts()

Состояние
failed        197719
successful    133956
canceled       38779
undefined       3562
live            2799
suspended       1846
Name: count, dtype: int64

In [8]:
data = data[data['Состояние'].isin(['failed', 'successful'])]
data['Состояние'].value_counts()

Состояние
failed        197719
successful    133956
Name: count, dtype: int64

## Задача сводиться к классификации и регрессии
### Что берем за таргет? 2 варианта:

- Будем решать задачу классификации, разметив объекты следующим образом: те проекты, у которых успешный статус, единичкой, а остальные ноликами

- Можем предсказывать просто собранное количество денег, применять модель, а потом уже смотреть, нужная ли сумма получилось. Тогда мы решаем задачу регрессии.


In [9]:
data.loc[(data['Состояние'] == 'failed'), 'target'] = 0
data.loc[(data['Состояние'] == 'successful'), 'target'] = 1
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах,target
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.00,600.0,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.00,10000.0,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.00,1400.0,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,successful,86,US,10030.88,10000.0,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,failed,0,US,0.00,10000.0,0.0


In [10]:
data = data.drop('Состояние', axis=1)

## Регрессия

In [11]:
data = data.rename({'Собрано в долларах':'target2'}, axis=1)
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0


In [12]:
data['Дедлайн'] = pd.to_datetime(data['Дедлайн'])
data['Дата публикации'] = pd.to_datetime(data['Дата публикации'])

In [13]:
data['Срок'] = (data['Дедлайн'] - data['Дата публикации']).dt.days


In [14]:
data['Год публикации'] = data['Дата публикации'].dt.year

In [15]:
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target,Срок,Год публикации
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0,31,2012
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0,29,2016
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0,29,2015
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0,44,2013
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0,29,2012


### Чтобы получить матрицу объектов, зачастую нужно обработать сырые данные, то есть извлечь из имеющихся таблиц признаки там, где они не даны явно

In [16]:
macro = pd.read_excel('macrofeatures.xlsx', engine='openpyxl')
macro.head()

,Unnamed: 0,Close_brent,Close_sugar,Close_cereals,Close_index_moex,Close_index_moex_10,Close_index_RGBI,Close_index_RTS_oil_and_gas,Close_index_RTS_metallurgy,Close_index_RTS_consumer_sector,Close_index_RTS_telecom,Close_index_RTS_finance,Close_index_RTS_transport,Close_index_RTS_chemicals,Close_index_RTS_broad_market,Close_index_RTS_electricity,dlk_cob_date
0,0,34.41,13.97,442.75,1797.27,3940.81,125.59,123.40,111.97,196.55,70.17,140.57,27.06,177.38,530.59,32.49,2016-02-24
1,1,35.06,14.24,445.25,1803.89,3977.35,126.44,124.22,112.51,198.03,70.56,142.64,27.43,179.48,536.20,33.07,2016-02-25
2,2,35.13,14.00,443.25,1816.73,4027.23,126.90,125.38,113.44,200.13,71.94,145.45,28.06,181.56,544.73,33.55,2016-02-26
3,3,36.64,14.36,445.00,1840.17,4084.24,126.87,126.69,114.66,200.32,72.41,147.22,28.49,186.76,552.82,34.41,2016-02-29
4,4,36.60,14.39,438.50,1844.17,4087.06,127.78,129.72,117.09,204.30,74.26,150.04,30.12,190.67,565.45,34.96,2016-03-01


In [17]:
macro = macro[['Close_brent', 'dlk_cob_date']].drop_duplicates()
macro['dlk_cob_date'] = pd.to_datetime(macro['dlk_cob_date'])

data.head()
data = pd.merge(data, macro,
         left_on=['Дата публикации'],
         right_on=['dlk_cob_date'],
         how='left')

In [18]:
data = data.sort_values('Дата публикации')
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target,Срок,Год публикации,Close_brent,dlk_cob_date
176128,Grace Jones Does Not Give A F$#% T-Shirt (limi...,Fashion,Fashion,USD,2009-05-31,2009-04-21 21:02:48,30,US,625.0,1000.0,0.0,39,2009,NaN,NaT
241929,CRYSTAL ANTLERS UNTITLED MOVIE,Shorts,Film & Video,USD,2009-07-20,2009-04-23 00:07:53,3,US,22.0,80000.0,0.0,87,2009,NaN,NaT
244460,drawing for dollars,Illustration,Art,USD,2009-05-03,2009-04-24 21:52:03,3,US,35.0,20.0,1.0,8,2009,NaN,NaT
80845,Offline Wikipedia iPhone app,Software,Technology,USD,2009-07-14,2009-04-25 17:36:21,25,US,145.0,99.0,1.0,79,2009,NaN,NaT
181197,Pantshirts,Fashion,Fashion,USD,2009-05-26,2009-04-27 14:10:39,10,US,387.0,1900.0,0.0,28,2009,NaN,NaT


In [19]:
data['Close_brent'] = data['Close_brent'].fillna(data['Close_brent'].mean())

In [20]:
data = data.drop(['Дедлайн', 'Дата публикации', 'dlk_cob_date'], axis=1)


In [21]:
data = data.drop(['Название', 'Страна', 'Инвесторов'], axis=1)

In [22]:
data = pd.concat((data, pd.get_dummies(data['Валюта'])), axis=1)
data = data.drop(['Валюта'], axis=1)

In [23]:
data = data.drop(['AUD'], axis=1)

In [24]:
data = pd.concat((data, pd.get_dummies(data['Главная категория'])), axis=1)
data = data.drop(['Главная категория'], axis=1)

In [25]:
data = data.drop(['Games'], axis=1)

In [26]:
data['Категория'] = data['Категория'].map(data.groupby(['Категория'])['target2'].mean())

In [27]:
data.head()

,Категория,target2,Цель в долларах,target,Срок,Год публикации,Close_brent,CAD,CHF,DKK,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,6035.989239,625.0,1000.0,0.0,39,2009,48.505,False,False,False,...,False,True,False,False,False,False,False,False,False,False
241929,3591.033473,22.0,80000.0,0.0,87,2009,48.505,False,False,False,...,False,False,True,False,False,False,False,False,False,False
244460,3661.424550,35.0,20.0,1.0,8,2009,48.505,False,False,False,...,False,False,False,False,False,False,False,False,False,False
80845,4321.245721,145.0,99.0,1.0,79,2009,48.505,False,False,False,...,False,False,False,False,False,False,False,False,True,False
181197,6035.989239,387.0,1900.0,0.0,28,2009,48.505,False,False,False,...,False,True,False,False,False,False,False,False,False,False


## Определимся с таргетом

In [28]:
X = data.drop(['target2', 'target'], axis=1)
Y = data['target2']


### sklearn

In [29]:
from sklearn.linear_model import LinearRegression

In [30]:
model = LinearRegression()
model.fit(X, Y)
X['Предсказание'] = model.predict(X)
X.head()

,Категория,Цель в долларах,Срок,Год публикации,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater,Предсказание
176128,6035.989239,1000.0,39,2009,48.505,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,3125.878031
241929,3591.033473,80000.0,87,2009,48.505,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,5117.839135
244460,3661.424550,20.0,8,2009,48.505,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,-1680.562044
80845,4321.245721,99.0,79,2009,48.505,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,4935.969700
181197,6035.989239,1900.0,28,2009,48.505,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,2183.766185


## Оценка модели

### MSE

In [31]:
(((X['Предсказание'] - Y)**2).mean())**(1/2)

np.float64(95926.36316736735)

### MAE

In [32]:
abs(X['Предсказание'] - Y).mean()

np.float64(13854.013934531977)

### OLS

In [43]:
from sklearn.linear_model import LinearRegression

In [53]:
model = LinearRegression()
model.fit(X, Y)
for column, coef in zip(X.columns, model.coef_):
    print(column, coef)
print(model.intercept_)

Категория 0.9918417335082822
Цель в долларах 0.0004601645870915237
Срок 85.68418124997802
Год публикации 880.0363179147823
Close_brent -2144.233728093301
CAD 473.79081724954324
CHF 8893.767874523091
DKK 542.9292904435097
EUR 479.96422938312065
GBP 2799.1804823789644
HKD -2065.455752878716
JPY -8469.178818142122
MXN -3973.270780039663
NOK -1161.8150624146745
NZD -382.41222006275086
SEK 2118.1704360497242
SGD -1319.870956077118
USD 7038.129637152052
Art -91.01411534112198
Comics -506.5142838268748
Crafts -629.930660861985
Dance -366.19887024712625
Design -232.50940249313652
Fashion -296.42697805312787
Film & Video -28.650411103639744
Food -724.891461032874
Journalism -475.3155389281199
Music -273.5324864722393
Photography 220.80447234905964
Publishing -362.9877971492459
Technology -212.5337669746021
Theater 260.0854009253156
-1676931.6196015282


In [65]:
X['constant'] = 1
X = X.astype({col: int for col in X.select_dtypes(include=['bool']).columns})
X

,Категория,Цель в долларах,Срок,Год публикации,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater,constant
176128,6035.989239,1000.00,39,2009,48.505,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
241929,3591.033473,80000.00,87,2009,48.505,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
244460,3661.424550,20.00,8,2009,48.505,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
80845,4321.245721,99.00,79,2009,48.505,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
181197,6035.989239,1900.00,28,2009,48.505,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247730,7635.064778,35.98,2,2017,48.505,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
318187,38415.722876,271.03,4,2017,48.505,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
264474,6098.303122,200.00,3,2017,48.505,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
293634,38415.722876,250.00,1,2017,48.505,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [66]:
xxt = np.dot(X.T, X)
xxt_inv = np.linalg.inv(xxt)
xxt_inv_xt = np.dot(xxt_inv, X.T)
final_beta = np.dot(xxt_inv_xt, Y)
final_beta


array([ 9.91841741e-01,  4.60164587e-04,  8.56841802e+01,  8.80036318e+02,
       -2.14423845e+03,  4.73790817e+02,  8.89376787e+03,  5.42929291e+02,
        4.79964229e+02,  2.79918048e+03, -2.06545575e+03, -8.46917882e+03,
       -3.97327078e+03, -1.16181506e+03, -3.82412220e+02,  2.11817044e+03,
       -1.31987096e+03,  7.03812964e+03, -9.10141156e+01, -5.06514284e+02,
       -6.29930661e+02, -3.66198871e+02, -2.32509402e+02, -2.96426977e+02,
       -2.86504115e+01, -7.24891461e+02, -4.75315539e+02, -2.73532487e+02,
        2.20804472e+02, -3.62987798e+02, -2.12533767e+02,  2.60085401e+02,
       -1.67693139e+06])

In [68]:
for column, coef in zip(X.columns, final_beta):
    print(column, coef)

Категория 0.9918417411881436
Цель в долларах 0.0004601645868189676
Срок 85.68418024613175
Год публикации 880.0363176822208
Close_brent -2144.2384472704452
CAD 473.7908172264759
CHF 8893.76787471839
DKK 542.9292905451947
EUR 479.9642287411223
GBP 2799.18048231655
HKD -2065.455752541588
JPY -8469.17881789209
MXN -3973.2707794434064
NOK -1161.815062272904
NZD -382.4122200612977
SEK 2118.1704362210517
SGD -1319.8709557969923
USD 7038.129636871795
Art -91.01411556115988
Comics -506.5142839212784
Crafts -629.9306608160877
Dance -366.19887055779014
Design -232.50940247415835
Fashion -296.4269774805461
Film & Video -28.650411471872324
Food -724.8914610852808
Journalism -475.31553898872363
Music -273.532486829809
Photography 220.80447210249906
Publishing -362.98779765895466
Technology -212.53376685934253
Theater 260.0854005196132
constant -1676931.3902270475
